# Cross-dataset 00 · Audit, matched-channel D2 stores, compatibility, freeze

**Rules of this experiment** (docs/cross_dataset/design.md): outputs only in `<outputs>/CROSS_D3_D2_19CH_v1/`;
the release (`FINAL_v1/`), amendment A2 (running Exp 8a), the D3 stores, their lock, pins and decisions are only
read. Source-only training; the target is BLINDED (labels masked) when it is scored; target labels are read only by
the evaluation in notebook 03, after the campaign is frozen. Nothing is selected with target results.

**Part A (no authorization needed; nothing is built):** the verified 19-electrode mapping (D2 EDF channel number ->
D3 label, T3=T7, T4=T8, T5=P7, T6=P8), the layout file hash, the shared head geometry, the D2 EDF headers (technical
fields only), the D3 stores (read only), the inherited atlas / graph decisions, and a recording-level inspection of
D2 (durations, selection, candidate windows; no source model, no graph).

**Part B (only with the author's written authorization):** the matched-channel D2 stores (LCMV and wMNE, modes B and
A, lobes12) from the 23 original resting-state EDFs, their validation, the compatibility report, the label-blind
shift diagnostics and D2 QC, and the harmonization manifest.

**Part C (author):** the frozen campaign (configs/cross_dataset/campaign.yaml) is recorded before any training.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything this notebook writes goes to
`<outputs>/CROSS_D3_D2_19CH_v1/` on Drive (never to the temporary Colab disk, and never into `FINAL_v1/`, the
amendment A2 folder or the D3 stores, which are only read).
 A CPU runtime is enough.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move THIS EXPERIMENT's code pin deliberately
                      # (<outputs>/CROSS_D3_D2_19CH_v1/CODE_COMMIT.txt; the release and A2 pins are never used or moved here).
BRANCH = 'claude/cross-dataset-d3-d2-19ch'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--namespace', 'cross'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh as _sh

def sh(cmd, cwd=None):  # every script runs with the file backend: Colab's inline backend breaks figures in scripts
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs (READ ONLY here):', FINAL)
CROSS = OUT + '/CROSS_D3_D2_19CH_v1'
print('cross-dataset outputs (the only place this notebook writes):', CROSS)

## Numerical environment (the Python / library versions of the completed Exp 7 units)

The cross-dataset fits must be comparable with Exp 0-8a, so every script runs with the interpreter that reproduces the
versions recorded by a completed Exp 7 unit (python, numpy, scipy, pandas, scikit-learn, torch); if this kernel
differs, `/content/restore<python digits>` is built with `uv` and verified (temporary Colab disk only). Leave
`ACCEPT_ENVIRONMENT_DIFFERENCE = False`.

In [ ]:
ACCEPT_ENVIRONMENT_DIFFERENCE = False
from adgraph import numerical_env
REFERENCE = numerical_env.reference_libraries(OUT, 'exp7_earm_gnn')
PY = numerical_env.ensure_interpreter(REFERENCE, repo='/content/repo', base='/content',
                                      accept_difference=ACCEPT_ENVIRONMENT_DIFFERENCE, run=sh)
DIFF = numerical_env.differences(REFERENCE, numerical_env.interpreter_versions(PY, REFERENCE))
assert ACCEPT_ENVIRONMENT_DIFFERENCE or not DIFF, f'numerical environment differs from Exp 7: {DIFF}'
def cross(args):   # every step of this experiment: scripts/cross_dataset.py with the verified interpreter
    sh(f"{PY} scripts/cross_dataset.py {args}", cwd='/content/repo')

## Part A1 · Mapping, layout, geometry, EDF headers, D3 stores, inherited decisions (read only)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
cross('audit')
display(pd.read_csv(CROSS + '/audit/channel_mapping_audit.csv'))
display(pd.read_csv(CROSS + '/audit/d2_edf_headers.csv'))
print('D2 EDF annotation texts (recording-condition evidence, decision D6):',
      json.load(open(CROSS + '/audit/audit.json'))['d2_edf_annotations'])
display(pd.read_csv(CROSS + '/audit/demographics.csv').round(1))          # age / sex by diagnosis (descriptive)
display(pd.read_csv(CROSS + '/audit/age_overlap_feasibility.csv'))
print(json.dumps(json.load(open(CROSS + '/audit/d2_filter_response.json'))['magnitude_db'], indent=1))

## Part A2 · Recording-level inspection (label-blind; no source model, no graph)

`inspect-d2-units`: every EDF channel of every D2 recording (unit, physical / digital range, the amplitude MNE
returns after the band-pass); recordings with mixed `V` / `uV` headers are listed channel by channel; an implausible
SELECTED channel fails. `inspect-d2`: every D2 recording processed exactly as the store will be, WITHOUT and WITH the
acquisition-specific 50 Hz notch (same crops and pieces): 50 Hz power, the 100 Hz harmonic, residual narrowband peaks,
covariance change (the LCMV input), accepted windows and the power change in every analysis band; `diag_*` columns
are the first audit's diagnostic notch on resampled segments (not the preprocessing notch). `inspect-d3-line`: the
D3 line-noise audit (D3 is not notched).

In [ ]:
FINDINGS = {}
for step in ('inspect-d2-units', 'inspect-d2', 'inspect-d3-line'):   # every audit runs; findings are collected
    try:
        cross(step)
    except RuntimeError as exc:
        FINDINGS[step] = str(exc)
units = pd.read_csv(CROSS + '/audit/d2_units.csv')
display(units[(units['selected_as'].notna()) & (units['finding'] != 'ok')])
display(pd.read_csv(CROSS + '/audit/d2_inspection.csv').filter(
    regex='key|selected_duration|windows_accepted|flip|line_ratio|line_variance|harmonic|covariance|band_power|peaks_after|mixed|status').round(3))
print(json.dumps(json.load(open(CROSS + '/audit/d2_notch_response.json'))['cascade_db'], indent=1))
if FINDINGS:
    raise RuntimeError(f'Part A2 findings (send them; nothing is built): {sorted(FINDINGS)}')

## Part B · Matched-channel D2 stores (STOP HERE unless the author has authorized it in writing)

Set `AUTHORIZED_BY` to the author's name only after the written authorization; it is recorded once in
`CROSS_D3_D2_19CH_v1/authorizations.json`. A two-participant smoke build (`--limit 2`) runs and is validated first.

In [ ]:
AUTHORIZED_BY = ''          # e.g. 'Sami (written authorization 2026-..-..)'; empty = stop here
if not AUTHORIZED_BY:
    print('Not authorized: the D2 stores are not built. Review Part A and docs/cross_dataset/design.md first.')
else:
    cross(f'authorize --stage build_d2 --by "{AUTHORIZED_BY}"')
    cross('build-d2 --inverse lcmv --limit 2')
    cross('validate-d2 --inverse lcmv --limit 2')
    for inverse in ('lcmv', 'wmne'):
        cross(f'build-d2 --inverse {inverse}')
        cross(f'validate-d2 --inverse {inverse}')
    for inverse in ('lcmv', 'wmne'):
        for mode in ('B', 'A'):
            cross(f'compat --inverse {inverse} --mode {mode}')

## Part C · Freeze the campaign (author; before any training)

In [ ]:
FREEZE_CAMPAIGN = False     # True only after reviewing Part A / B outputs and the design document
if FREEZE_CAMPAIGN:
    cross('freeze')
else:
    print('campaign not frozen; nothing can be trained or evaluated')